In [56]:
import pandas as pd
import ast
import re
import pickle

In [26]:
chennai_df = pd.read_excel(r"E:\GUVI Projects\Cardehko\Dataset\chennai_cars.xlsx")

In [27]:
chennai_df.head()

,new_car_detail,new_car_overview,new_car_feature,new_car_specs,car_links
0,"{'it': 0, 'ft': 'Petrol', 'bt': 'SUV', 'km': '...","{'heading': 'Car overview', 'top': [{'key': 'R...","{'heading': 'Features', 'top': [{'value': 'Pow...","{'heading': 'Specifications', 'top': [{'key': ...",https://www.cardekho.com/used-car-details/used...
1,"{'it': 0, 'ft': 'Petrol', 'bt': 'Minivans', 'k...","{'heading': 'Car overview', 'top': [{'key': 'R...","{'heading': 'Features', 'top': [{'value': 'Low...","{'heading': 'Specifications', 'top': [{'key': ...",https://www.cardekho.com/buy-used-car-details/...
2,"{'it': 0, 'ft': 'Petrol', 'bt': 'SUV', 'km': '...","{'heading': 'Car overview', 'top': [{'key': 'R...","{'heading': 'Features', 'top': [{'value': 'Pow...","{'heading': 'Specifications', 'top': [{'key': ...",https://www.cardekho.com/used-car-details/used...
3,"{'it': 0, 'ft': 'Petrol', 'bt': 'Hatchback', '...","{'heading': 'Car overview', 'top': [{'key': 'R...","{'heading': 'Features', 'top': [{'value': 'Pow...","{'heading': 'Specifications', 'top': [{'key': ...",https://www.cardekho.com/buy-used-car-details/...
4,"{'it': 0, 'ft': 'Petrol', 'bt': 'Hatchback', '...","{'heading': 'Car overview', 'top': [{'key': 'R...","{'heading': 'Features', 'top': [{'value': 'Pow...","{'heading': 'Specifications', 'top': [{'key': ...",https://www.cardekho.com/used-car-details/used...


In [28]:
chennai_df['new_car_detail'].head()

0    {'it': 0, 'ft': 'Petrol', 'bt': 'SUV', 'km': '...
1    {'it': 0, 'ft': 'Petrol', 'bt': 'Minivans', 'k...
2    {'it': 0, 'ft': 'Petrol', 'bt': 'SUV', 'km': '...
3    {'it': 0, 'ft': 'Petrol', 'bt': 'Hatchback', '...
4    {'it': 0, 'ft': 'Petrol', 'bt': 'Hatchback', '...
Name: new_car_detail, dtype: object

In [29]:
# Extract data from each structured column
car_details_df = pd.json_normalize(chennai_df['new_car_detail'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)) 

In [30]:
car_details_df.columns

Index(['it', 'ft', 'bt', 'km', 'transmission', 'ownerNo', 'owner', 'oem',
       'model', 'modelYear', 'centralVariantId', 'variantName', 'price',
       'priceActual', 'priceSaving', 'priceFixedText', 'trendingText.imgUrl',
       'trendingText.heading', 'trendingText.desc'],
      dtype='object')

In [31]:
# Function to extract 'top' items from the car overview, including icons
def extract_overview_data(overview):
    if isinstance(overview, str):
        overview = ast.literal_eval(overview)
    
    # Extract key, value, and icon
    extracted_data = {}
    for item in overview.get('top', []):
        key = item['key']
        extracted_data[f"{key}_value"] = item['value']
        extracted_data[f"{key}_icon"] = item.get('icon', '')  # Use empty string if icon is missing
    
    return extracted_data

# Apply the extraction function to the 'new_car_overview' column
car_overview_df = chennai_df['new_car_overview'].apply(extract_overview_data).apply(pd.Series)

# Display the extracted DataFrame
print(car_overview_df.columns)

Index(['Registration Year_value', 'Registration Year_icon',
       'Insurance Validity_value', 'Insurance Validity_icon',
       'Fuel Type_value', 'Fuel Type_icon', 'Seats_value', 'Seats_icon',
       'Kms Driven_value', 'Kms Driven_icon', 'RTO_value', 'RTO_icon',
       'Ownership_value', 'Ownership_icon', 'Engine Displacement_value',
       'Engine Displacement_icon', 'Transmission_value', 'Transmission_icon',
       'Year of Manufacture_value', 'Year of Manufacture_icon'],
      dtype='object')


In [32]:
# Function to extract features from the car feature data
def extract_features(features):
    if isinstance(features, str):
        features = ast.literal_eval(features)
    
    top_features = [item['value'] for item in features.get('top', [])]
    common_icon = features.get('commonIcon', '')
    
    return top_features, common_icon

# Apply the function and create a DataFrame
car_features_df = chennai_df['new_car_feature'].apply(extract_features).apply(pd.Series)
car_features_df.columns = ['Top_Features', 'Common_Icon']

# Display the extracted DataFrame
print(car_features_df.columns)

Index(['Top_Features', 'Common_Icon'], dtype='object')


In [33]:
# Function to extract specifications from a structured dictionary
def extract_specifications(specifications):
    if isinstance(specifications, str):
        specifications = ast.literal_eval(specifications)
        
    top_specs = {item['key']: item['value'] for item in specifications.get('top', [])}
    
    detailed_specs = {}
    for category in specifications.get('data', []):
        for item in category.get('list', []):
            detailed_specs[item['key']] = item['value']
    
    return {**top_specs, **detailed_specs, 'Common_Icon': specifications.get('commonIcon', '')}

# **top_specs unpacks all key-value pairs from the top_specs dictionary.
# **detailed_specs unpacks all key-value pairs from the detailed_specs dictionary.

# Apply the extraction function to the 'new_car_specifications' column
car_specs_df = chennai_df['new_car_specs'].apply(extract_specifications).apply(pd.Series)

# Display the extracted DataFrame
print(car_specs_df.columns)

Index(['Engine', 'Max Power', 'Torque', 'Wheel Size', 'Seats', 'Color',
       'Engine Type', 'Displacement', 'Max Torque', 'No of Cylinder',
       'Values per Cylinder', 'Fuel Suppy System', 'Turbo Charger', 'Length',
       'Width', 'Height', 'Wheel Base', 'Kerb Weight', 'Gear Box',
       'Drive Type', 'Seating Capacity', 'Steering Type', 'Front Brake Type',
       'Rear Brake Type', 'Tyre Type', 'Alloy Wheel Size', 'No Door Numbers',
       'Cargo Volumn', 'Common_Icon', 'Mileage', 'Value Configuration',
       'Compression Ratio', 'Super Charger', 'Front Tread', 'Rear Tread',
       'Gross Weight', 'Turning Radius', 'Top Speed', 'Acceleration',
       'BoreX Stroke', 'Ground Clearance Unladen'],
      dtype='object')


In [34]:
car_details_df

,it,ft,bt,km,transmission,ownerNo,owner,oem,model,modelYear,centralVariantId,variantName,price,priceActual,priceSaving,priceFixedText,trendingText.imgUrl,trendingText.heading,trendingText.desc
0,0,Petrol,SUV,"20,000",Automatic,1,1st Owner,Kia,Kia Sonet,2022,8654,Turbo DCT Anniversary Edition,₹ 11.50 Lakh,,,None,https://stimg.cardekho.com/used-cars/common/ic...,Trending Car!,High chances of sale in next 6 days
1,0,Petrol,Minivans,"20,687",Manual,1,1st Owner,Maruti,Maruti Eeco,2015,4025,7 Seater Standard BSIV,₹ 4.15 Lakh,,,None,https://stimg.cardekho.com/used-cars/common/ic...,Trending Car!,High chances of sale in next 6 days
2,0,Petrol,SUV,"30,000",Manual,1,1st Owner,Nissan,Nissan Magnite,2021,8135,Turbo XV Premium BSVI,₹ 7.50 Lakh,,,None,https://stimg.cardekho.com/used-cars/common/ic...,Trending Car!,High chances of sale in next 6 days
3,0,Petrol,Hatchback,"59,247",Manual,1,1st Owner,Hyundai,Hyundai i10,2015,1579,Sportz 1.1L,₹ 3.98 Lakh,,,None,https://stimg.cardekho.com/used-cars/common/ic...,Trending Car!,High chances of sale in next 6 days
4,0,Petrol,Hatchback,"50,000",Manual,1,1st Owner,Honda,Honda Jazz,2015,1341,1.2 VX i VTEC,₹ 5.50 Lakh,,,None,https://stimg.cardekho.com/used-cars/common/ic...,Trending Car!,High chances of sale in next 6 days
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1414,0,Petrol,Hatchback,"42,891",Automatic,1,1st Owner,Maruti,Maruti Swift,2018,6190,AMT VXI,₹ 6.20 Lakh,,,None,https://stimg.cardekho.com/used-cars/common/ic...,Trending Car!,High chances of sale in next 6 days
1415,0,Diesel,SUV,"59,100",Manual,1,1st Owner,Renault,Renault Duster,2018,5050,85PS Diesel RxZ,₹ 8.90 Lakh,,,None,https://stimg.cardekho.com/used-cars/common/ic...,Trending Car!,High chances of sale in next 6 days
1416,0,Petrol,SUV,"55,913",Automatic,1,1st Owner,Tata,Tata Nexon,2018,6364,1.2 Revotron XZA Plus,₹ 8.01 Lakh,,,None,https://stimg.cardekho.com/used-cars/common/ic...,Trending Car!,High chances of sale in next 6 days
1417,0,Diesel,SUV,"65,000",Automatic,1,1st Owner,Volkswagen,Volkswagen Tiguan,2017,5849,2.0 TDI Highline,₹ 20.50 Lakh,,,None,https://stimg.cardekho.com/used-cars/common/ic...,Trending Car!,High chances of sale in next 6 days


In [35]:
# Create a DataFrame from the car_links
car_links_df = chennai_df['car_links'].reset_index(drop=True)

# Combine all DataFrames into one
chennai_combined_df = pd.concat([
    car_details_df.reset_index(drop=True),
    car_overview_df.reset_index(drop=True),
    car_features_df.reset_index(drop=True),
    car_specs_df.reset_index(drop=True),
    car_links_df], axis=1)

# Add a new column named 'city' with the value 'chennai'
chennai_combined_df.insert(0, 'city', 'chennai')

# Save the final combined DataFrame to a CSV file
output_file = 'chennai_cars_final.csv'
chennai_combined_df.to_csv(output_file, index=False)

print(f"Structured data saved to {output_file}")

Structured data saved to chennai_cars_final.csv


In [36]:
chennai_combined_df

,city,it,ft,bt,km,transmission,ownerNo,owner,oem,model,...,Super Charger,Front Tread,Rear Tread,Gross Weight,Turning Radius,Top Speed,Acceleration,BoreX Stroke,Ground Clearance Unladen,car_links
0,chennai,0,Petrol,SUV,"20,000",Automatic,1,1st Owner,Kia,Kia Sonet,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.cardekho.com/used-car-details/used...
1,chennai,0,Petrol,Minivans,"20,687",Manual,1,1st Owner,Maruti,Maruti Eeco,...,No,1280mm,1290mm,1540kg,4.5 metres,145 Kmph,15.7 Seconds,NaN,NaN,https://www.cardekho.com/buy-used-car-details/...
2,chennai,0,Petrol,SUV,"30,000",Manual,1,1st Owner,Nissan,Nissan Magnite,...,No,NaN,NaN,NaN,5.0,NaN,11.7,72.2 x 81.3,NaN,https://www.cardekho.com/used-car-details/used...
3,chennai,0,Petrol,Hatchback,"59,247",Manual,1,1st Owner,Hyundai,Hyundai i10,...,No,1400mm,1385mm,NaN,4.7 metres,165 Kmph,14.3 Seconds,NaN,NaN,https://www.cardekho.com/buy-used-car-details/...
4,chennai,0,Petrol,Hatchback,"50,000",Manual,1,1st Owner,Honda,Honda Jazz,...,No,NaN,NaN,NaN,5.1 meters,172 Kmph,13.7 Seconds,NaN,NaN,https://www.cardekho.com/used-car-details/used...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1414,chennai,0,Petrol,Hatchback,"42,891",Automatic,1,1st Owner,Maruti,Maruti Swift,...,No,1530mm,1530mm,1315kg,4.8 Meters,NaN,NaN,NaN,NaN,https://www.cardekho.com/used-car-details/used...
1415,chennai,0,Diesel,SUV,"59,100",Manual,1,1st Owner,Renault,Renault Duster,...,No,1560mm,1567mm,1777kg,5.2 metres,156 Kmph,13.9 Seconds,NaN,NaN,https://www.cardekho.com/used-car-details/used...
1416,chennai,0,Petrol,SUV,"55,913",Automatic,1,1st Owner,Tata,Tata Nexon,...,No,1540 mm,1530 mm,NaN,5.1m,154.19 kmph,NaN,77x85.8,209 mm,https://www.cardekho.com/used-car-details/used...
1417,chennai,0,Diesel,SUV,"65,000",Automatic,1,1st Owner,Volkswagen,Volkswagen Tiguan,...,No,1578mm,1568mm,2250kg,5.75meters,NaN,NaN,74.5 x 81 mm,NaN,https://www.cardekho.com/used-car-details/used...


In [37]:
Bangalore_df = pd.read_excel(r"E:\GUVI Projects\Cardehko\Dataset\bangalore_cars.xlsx")
Bangalore_df.shape

(1481, 5)

In [38]:
# Extract data from each structured column
car_details_df = pd.json_normalize(Bangalore_df['new_car_detail'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)) 

# Function to extract 'top' items from the car overview, including icons
def extract_overview_data(overview):
    if isinstance(overview, str):
        overview = ast.literal_eval(overview)
    
    # Extract key, value, and icon
    extracted_data = {}
    for item in overview.get('top', []):
        key = item['key']
        extracted_data[f"{key}_value"] = item['value']
        extracted_data[f"{key}_icon"] = item.get('icon', '')  # Use empty string if icon is missing
    
    return extracted_data

# Apply the extraction function to the 'new_car_overview' column
car_overview_df = Bangalore_df['new_car_overview'].apply(extract_overview_data).apply(pd.Series)

# Function to extract features from the car feature data
def extract_features(features):
    if isinstance(features, str):
        features = ast.literal_eval(features)
    
    top_features = [item['value'] for item in features.get('top', [])]
    common_icon = features.get('commonIcon', '')
    
    return top_features, common_icon

# Apply the function and create a DataFrame
car_features_df = Bangalore_df['new_car_feature'].apply(extract_features).apply(pd.Series)
car_features_df.columns = ['Top_Features', 'Common_Icon']

# Function to extract specifications from a structured dictionary
def extract_specifications(specifications):
    if isinstance(specifications, str):
        specifications = ast.literal_eval(specifications)
        
    top_specs = {item['key']: item['value'] for item in specifications.get('top', [])}
    
    detailed_specs = {}
    for category in specifications.get('data', []):
        for item in category.get('list', []):
            detailed_specs[item['key']] = item['value']
    
    return {**top_specs, **detailed_specs, 'Common_Icon': specifications.get('commonIcon', '')}

# Apply the extraction function to the 'new_car_specifications' column
car_specs_df = Bangalore_df['new_car_specs'].apply(extract_specifications).apply(pd.Series)

# Create a DataFrame from the car_links
car_links_df = Bangalore_df['car_links'].reset_index(drop=True)

# Combine all DataFrames into one
Bangalore_combined_df = pd.concat([
    car_details_df.reset_index(drop=True),
    car_overview_df.reset_index(drop=True),
    car_features_df.reset_index(drop=True),
    car_specs_df.reset_index(drop=True),
    car_links_df], axis=1)

Bangalore_combined_df.insert(0, 'city', 'Bangalore')

# Save the final combined DataFrame to a CSV file
Bangalore_combined_df.to_csv(r'Bangalore_cars_final.csv')

In [39]:
Bangalore_combined_df.head()

,city,it,ft,bt,km,transmission,ownerNo,owner,oem,model,...,Top Speed,Acceleration,Tyre Type,No Door Numbers,Cargo Volumn,Common_Icon,Wheel Size,Alloy Wheel Size,Ground Clearance Unladen,car_links
0,Bangalore,0,Petrol,Hatchback,"1,20,000",Manual,3,3rd Owner,Maruti,Maruti Celerio,...,150 Kmph,15.05 Seconds,"Tubeless, Radial",5,235-litres,,NaN,NaN,NaN,https://www.cardekho.com/used-car-details/used...
1,Bangalore,0,Petrol,SUV,"32,706",Manual,2,2nd Owner,Ford,Ford Ecosport,...,NaN,NaN,"Tubeless,Radial",4,352-litres,,16,16,NaN,https://www.cardekho.com/buy-used-car-details/...
2,Bangalore,0,Petrol,Hatchback,"11,949",Manual,1,1st Owner,Tata,Tata Tiago,...,150 kmph,14.3 Seconds,Tubeless,5,242-litres,,14,14,NaN,https://www.cardekho.com/used-car-details/used...
3,Bangalore,0,Petrol,Sedan,"17,794",Manual,1,1st Owner,Hyundai,Hyundai Xcent,...,172km/hr,14.2 Seconds,"Tubeless,Radial",4,407-litres,,14,14,NaN,https://www.cardekho.com/buy-used-car-details/...
4,Bangalore,0,Diesel,SUV,"60,000",Manual,1,1st Owner,Maruti,Maruti SX4 S Cross,...,190 Kmph,12 Seconds,"Tubeless,Radial",5,353-litres,,16,16,NaN,https://www.cardekho.com/used-car-details/used...


In [40]:
Bangalore_combined_df.shape

(1481, 84)

In [41]:
Delhi_df = pd.read_excel(r"E:\GUVI Projects\Cardehko\Dataset\delhi_cars.xlsx")
Delhi_df.shape

(1485, 5)

In [42]:
# Extract data from each structured column
car_details_df = pd.json_normalize(Delhi_df['new_car_detail'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)) 

# Function to extract 'top' items from the car overview, including icons
def extract_overview_data(overview):
    if isinstance(overview, str):
        overview = ast.literal_eval(overview)
    
    # Extract key, value, and icon
    extracted_data = {}
    for item in overview.get('top', []):
        key = item['key']
        extracted_data[f"{key}_value"] = item['value']
        extracted_data[f"{key}_icon"] = item.get('icon', '')  # Use empty string if icon is missing
    
    return extracted_data

# Apply the extraction function to the 'new_car_overview' column
car_overview_df = Delhi_df['new_car_overview'].apply(extract_overview_data).apply(pd.Series)

# Function to extract features from the car feature data
def extract_features(features):
    if isinstance(features, str):
        features = ast.literal_eval(features)
    
    top_features = [item['value'] for item in features.get('top', [])]
    common_icon = features.get('commonIcon', '')
    
    return top_features, common_icon

# Apply the function and create a DataFrame
car_features_df = Delhi_df['new_car_feature'].apply(extract_features).apply(pd.Series)
car_features_df.columns = ['Top_Features', 'Common_Icon']

# Function to extract specifications from a structured dictionary
def extract_specifications(specifications):
    if isinstance(specifications, str):
        specifications = ast.literal_eval(specifications)
        
    top_specs = {item['key']: item['value'] for item in specifications.get('top', [])}
    
    detailed_specs = {}
    for category in specifications.get('data', []):
        for item in category.get('list', []):
            detailed_specs[item['key']] = item['value']
    
    return {**top_specs, **detailed_specs, 'Common_Icon': specifications.get('commonIcon', '')}

# Apply the extraction function to the 'new_car_specifications' column
car_specs_df = Delhi_df['new_car_specs'].apply(extract_specifications).apply(pd.Series)

# Create a DataFrame from the car_links
car_links_df = Delhi_df['car_links'].reset_index(drop=True)

# Combine all DataFrames into one
Delhi_combined_df = pd.concat([
    car_details_df.reset_index(drop=True),
    car_overview_df.reset_index(drop=True),
    car_features_df.reset_index(drop=True),
    car_specs_df.reset_index(drop=True),
    car_links_df], axis=1)

Delhi_combined_df.insert(0, 'city', 'Delhi')

# Save the final combined DataFrame to a CSV file
Delhi_combined_df.to_csv(r'Delhi_cars_final.csv')

In [43]:

Delhi_combined_df.head()

,city,it,ft,bt,km,transmission,ownerNo,owner,oem,model,...,Turning Radius,Top Speed,Acceleration,Gross Weight,Front Tread,Rear Tread,BoreX Stroke,Compression Ratio,Ground Clearance Unladen,car_links
0,Delhi,0,Diesel,SUV,"10,000",Automatic,1,1st Owner,Kia,Kia Seltos,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.cardekho.com/used-car-details/used...
1,Delhi,0,Petrol,SUV,"57,437",Manual,2,2nd Owner,Hyundai,Hyundai Creta,...,5.3 metres,165 Kmph,10.5 Seconds,NaN,NaN,NaN,NaN,NaN,NaN,https://www.cardekho.com/buy-used-car-details/...
2,Delhi,0,Petrol,SUV,"8,000",Automatic,1,1st Owner,Mercedes-Benz,Mercedes-Benz GLC,...,NaN,217 Kmph,NaN,2360kg,NaN,NaN,NaN,NaN,NaN,https://www.cardekho.com/used-car-details/used...
3,Delhi,0,Petrol,Hatchback,"28,151",Manual,2,2nd Owner,Maruti,Maruti Swift,...,4.8,NaN,NaN,1335,1530,1530,NaN,NaN,NaN,https://www.cardekho.com/buy-used-car-details/...
4,Delhi,0,Petrol,SUV,"60,000",Manual,1,1st Owner,Hyundai,Hyundai Creta,...,5.3 metres,165 Kmph,10.5 Seconds,NaN,NaN,NaN,NaN,NaN,NaN,https://www.cardekho.com/used-car-details/used...


In [44]:
Delhi_combined_df.shape

(1485, 84)

In [46]:
Hyderabad_df = pd.read_excel(r"E:\GUVI Projects\Cardehko\Dataset\hyderabad_cars.xlsx")
Hyderabad_df.shape

(1483, 5)

In [47]:
# Extract data from each structured column
car_details_df = pd.json_normalize(Hyderabad_df['new_car_detail'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)) 

# Function to extract 'top' items from the car overview, including icons
def extract_overview_data(overview):
    if isinstance(overview, str):
        overview = ast.literal_eval(overview)
    
    # Extract key, value, and icon
    extracted_data = {}
    for item in overview.get('top', []):
        key = item['key']
        extracted_data[f"{key}_value"] = item['value']
        extracted_data[f"{key}_icon"] = item.get('icon', '')  # Use empty string if icon is missing
    
    return extracted_data

# Apply the extraction function to the 'new_car_overview' column
car_overview_df = Hyderabad_df['new_car_overview'].apply(extract_overview_data).apply(pd.Series)

# Function to extract features from the car feature data
def extract_features(features):
    if isinstance(features, str):
        features = ast.literal_eval(features)
    
    top_features = [item['value'] for item in features.get('top', [])]
    common_icon = features.get('commonIcon', '')
    
    return top_features, common_icon

# Apply the function and create a DataFrame
car_features_df = Hyderabad_df['new_car_feature'].apply(extract_features).apply(pd.Series)
car_features_df.columns = ['Top_Features', 'Common_Icon']

# Function to extract specifications from a structured dictionary
def extract_specifications(specifications):
    if isinstance(specifications, str):
        specifications = ast.literal_eval(specifications)
        
    top_specs = {item['key']: item['value'] for item in specifications.get('top', [])}
    
    detailed_specs = {}
    for category in specifications.get('data', []):
        for item in category.get('list', []):
            detailed_specs[item['key']] = item['value']
    
    return {**top_specs, **detailed_specs, 'Common_Icon': specifications.get('commonIcon', '')}

# Apply the extraction function to the 'new_car_specifications' column
car_specs_df = Hyderabad_df['new_car_specs'].apply(extract_specifications).apply(pd.Series)

# Create a DataFrame from the car_links
car_links_df = Hyderabad_df['car_links'].reset_index(drop=True)

# Combine all DataFrames into one
Hyderabad_combined_df = pd.concat([
    car_details_df.reset_index(drop=True),
    car_overview_df.reset_index(drop=True),
    car_features_df.reset_index(drop=True),
    car_specs_df.reset_index(drop=True),
    car_links_df], axis=1)

Hyderabad_combined_df.insert(0, 'city', 'Hyderabad')

# Save the final combined DataFrame to a CSV file
Hyderabad_combined_df.to_csv(r'Hyderabad_cars_final.csv')
Hyderabad_combined_df.shape


(1483, 84)

In [48]:
Kolkata_df = pd.read_excel(r"E:\GUVI Projects\Cardehko\Dataset\kolkata_cars.xlsx")
Kolkata_df.shape

(1381, 5)

In [49]:
# Extract data from each structured column
car_details_df = pd.json_normalize(Kolkata_df['new_car_detail'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)) 

# Function to extract 'top' items from the car overview, including icons
def extract_overview_data(overview):
    if isinstance(overview, str):
        overview = ast.literal_eval(overview)
    
    # Extract key, value, and icon
    extracted_data = {}
    for item in overview.get('top', []):
        key = item['key']
        extracted_data[f"{key}_value"] = item['value']
        extracted_data[f"{key}_icon"] = item.get('icon', '')  # Use empty string if icon is missing
    
    return extracted_data

# Apply the extraction function to the 'new_car_overview' column
car_overview_df = Kolkata_df['new_car_overview'].apply(extract_overview_data).apply(pd.Series)

# Function to extract features from the car feature data
def extract_features(features):
    if isinstance(features, str):
        features = ast.literal_eval(features)
    
    top_features = [item['value'] for item in features.get('top', [])]
    common_icon = features.get('commonIcon', '')
    
    return top_features, common_icon

# Apply the function and create a DataFrame
car_features_df = Kolkata_df['new_car_feature'].apply(extract_features).apply(pd.Series)
car_features_df.columns = ['Top_Features', 'Common_Icon']

# Function to extract specifications from a structured dictionary
def extract_specifications(specifications):
    if isinstance(specifications, str):
        specifications = ast.literal_eval(specifications)
        
    top_specs = {item['key']: item['value'] for item in specifications.get('top', [])}
    
    detailed_specs = {}
    for category in specifications.get('data', []):
        for item in category.get('list', []):
            detailed_specs[item['key']] = item['value']
    
    return {**top_specs, **detailed_specs, 'Common_Icon': specifications.get('commonIcon', '')}

# Apply the extraction function to the 'new_car_specifications' column
car_specs_df = Kolkata_df['new_car_specs'].apply(extract_specifications).apply(pd.Series)

# Create a DataFrame from the car_links
car_links_df = Kolkata_df['car_links'].reset_index(drop=True)

# Combine all DataFrames into one
Kolkata_combined_df = pd.concat([
    car_details_df.reset_index(drop=True),
    car_overview_df.reset_index(drop=True),
    car_features_df.reset_index(drop=True),
    car_specs_df.reset_index(drop=True),
    car_links_df], axis=1)

Kolkata_combined_df.insert(0, 'city', 'Kolkata')

# Save the final combined DataFrame to a CSV file
Kolkata_combined_df.to_csv(r'Kolkata_cars_final.csv')
Kolkata_combined_df.shape

(1381, 84)

In [50]:
Kolkata_combined_df.head()

,city,it,ft,bt,km,transmission,ownerNo,owner,oem,model,...,Tyre Type,Alloy Wheel Size,No Door Numbers,Common_Icon,Ground Clearance Unladen,Cargo Volumn,Compression Ratio,Acceleration,Top Speed,car_links
0,Kolkata,0,Petrol,Sedan,"70,000",Automatic,3,3rd Owner,Toyota,Toyota Camry,...,"Tubeless,Radial",17,4,,NaN,NaN,NaN,NaN,NaN,https://www.cardekho.com/used-car-details/used...
1,Kolkata,0,Petrol,Hatchback,"23,981",Manual,1,1st Owner,Datsun,Datsun RediGO,...,NaN,NaN,5,,185mm,222,NaN,NaN,NaN,https://www.cardekho.com/buy-used-car-details/...
2,Kolkata,0,Petrol,SUV,"7,100",Automatic,1,1st Owner,Renault,Renault Kiger,...,"Tubeless, Radial",NaN,5,,NaN,405,NaN,NaN,NaN,https://www.cardekho.com/used-car-details/used...
3,Kolkata,0,Petrol,Hatchback,"71,574",Manual,2,2nd Owner,Hyundai,Hyundai i20,...,"Tubeless,Radial",14,5,,NaN,295 Lit,:1,12.96 Sec,NaN,https://www.cardekho.com/buy-used-car-details/...
4,Kolkata,0,Diesel,SUV,"50,000",Automatic,2,2nd Owner,Audi,Audi Q3,...,"Tubeless,Radial",16,5,,NaN,460-litres,NaN,8.2 Seconds,212 Kmph,https://www.cardekho.com/used-car-details/used...


In [51]:
Jaipur_df = pd.read_excel(r"E:\GUVI Projects\Cardehko\Dataset\jaipur_cars.xlsx")
Jaipur_df.shape

(1120, 5)

In [52]:
# Extract data from each structured column
car_details_df = pd.json_normalize(Jaipur_df['new_car_detail'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)) 

# Function to extract 'top' items from the car overview, including icons
def extract_overview_data(overview):
    if isinstance(overview, str):
        overview = ast.literal_eval(overview)
    
    # Extract key, value, and icon
    extracted_data = {}
    for item in overview.get('top', []):
        key = item['key']
        extracted_data[f"{key}_value"] = item['value']
        extracted_data[f"{key}_icon"] = item.get('icon', '')  # Use empty string if icon is missing
    
    return extracted_data

# Apply the extraction function to the 'new_car_overview' column
car_overview_df = Jaipur_df['new_car_overview'].apply(extract_overview_data).apply(pd.Series)

# Function to extract features from the car feature data
def extract_features(features):
    if isinstance(features, str):
        features = ast.literal_eval(features)
    
    top_features = [item['value'] for item in features.get('top', [])]
    common_icon = features.get('commonIcon', '')
    
    return top_features, common_icon

# Apply the function and create a DataFrame
car_features_df = Jaipur_df['new_car_feature'].apply(extract_features).apply(pd.Series)
car_features_df.columns = ['Top_Features', 'Common_Icon']

# Function to extract specifications from a structured dictionary
def extract_specifications(specifications):
    if isinstance(specifications, str):
        specifications = ast.literal_eval(specifications)
        
    top_specs = {item['key']: item['value'] for item in specifications.get('top', [])}
    
    detailed_specs = {}
    for category in specifications.get('data', []):
        for item in category.get('list', []):
            detailed_specs[item['key']] = item['value']
    
    return {**top_specs, **detailed_specs, 'Common_Icon': specifications.get('commonIcon', '')}

# Apply the extraction function to the 'new_car_specifications' column
car_specs_df = Jaipur_df['new_car_specs'].apply(extract_specifications).apply(pd.Series)

# Create a DataFrame from the car_links
car_links_df = Jaipur_df['car_links'].reset_index(drop=True)

# Combine all DataFrames into one
Jaipur_combined_df = pd.concat([
    car_details_df.reset_index(drop=True),
    car_overview_df.reset_index(drop=True),
    car_features_df.reset_index(drop=True),
    car_specs_df.reset_index(drop=True),
    car_links_df], axis=1)

Jaipur_combined_df.insert(0, 'city', 'Jaipur')

# Save the final combined DataFrame to a CSV file
Jaipur_combined_df.to_csv(r'Jaipur_cars_final.csv')
Jaipur_combined_df.shape


(1120, 84)

In [53]:
Jaipur_combined_df.head()

,city,it,ft,bt,km,transmission,ownerNo,owner,oem,model,...,Tyre Type,Alloy Wheel Size,No Door Numbers,Cargo Volumn,Common_Icon,Gross Weight,BoreX Stroke,Compression Ratio,Ground Clearance Unladen,car_links
0,Jaipur,0,Diesel,Hatchback,"1,20,000",Manual,2,2nd Owner,Hyundai,Hyundai i20,...,Tubeless,16,5,295-litres,,NaN,NaN,NaN,NaN,https://www.cardekho.com/used-car-details/used...
1,Jaipur,0,Petrol,Hatchback,"66,951",Manual,1,1st Owner,Maruti,Maruti Swift,...,"Radial, Tubeless",NaN,5,268,,1335,NaN,NaN,NaN,https://www.cardekho.com/buy-used-car-details/...
2,Jaipur,0,Petrol,Hatchback,"80,000",Automatic,2,2nd Owner,Maruti,Maruti Celerio,...,"Tubeless, Radial",NaN,5,235-litres,,1250kg,73 X 82 mm,11.0:1,NaN,https://www.cardekho.com/used-car-details/used...
3,Jaipur,0,Petrol,Hatchback,"44,392",Manual,1,1st Owner,Hyundai,Hyundai Grand i10,...,Tubeless,NaN,5,256,,NaN,NaN,NaN,NaN,https://www.cardekho.com/buy-used-car-details/...
4,Jaipur,0,Petrol,Hatchback,"40,000",Automatic,1,1st Owner,Maruti,Maruti Wagon R,...,Tubeless Tyres,NaN,5,180-liters,,1350kg,69 x 72 mm,NaN,NaN,https://www.cardekho.com/used-car-details/used...


In [54]:
# Reset the index for each DataFrame to ensure unique index
chennai_combined_df = chennai_combined_df.reset_index(drop=True)
Bangalore_combined_df = Bangalore_combined_df.reset_index(drop=True)
Delhi_combined_df = Delhi_combined_df.reset_index(drop=True)
Hyderabad_combined_df = Hyderabad_combined_df.reset_index(drop=True)
Jaipur_combined_df = Jaipur_combined_df.reset_index(drop=True)
Kolkata_combined_df = Kolkata_combined_df.reset_index(drop=True)

# Ensure that all DataFrames have the same columns
common_columns = list(set(chennai_combined_df.columns).intersection(
    Bangalore_combined_df.columns,
    Delhi_combined_df.columns,
    Hyderabad_combined_df.columns,
    Jaipur_combined_df.columns,
    Kolkata_combined_df.columns
))

# Filter each DataFrame to include only the common columns
chennai_combined_df = chennai_combined_df[common_columns]
Bangalore_combined_df = Bangalore_combined_df[common_columns]
Delhi_combined_df = Delhi_combined_df[common_columns]
Hyderabad_combined_df = Hyderabad_combined_df[common_columns]
Jaipur_combined_df = Jaipur_combined_df[common_columns]
Kolkata_combined_df = Kolkata_combined_df[common_columns]

# Concatenate all city DataFrames
all_city_cars_df = pd.concat([chennai_combined_df, Bangalore_combined_df, Delhi_combined_df, Hyderabad_combined_df,
                              Jaipur_combined_df, Kolkata_combined_df], ignore_index=True)

# Save the combined DataFrame to a CSV file
all_city_cars_df.to_csv('all_city_cars.csv', index=False)

print("Data for all cities saved to 'all_city_cars.csv'")

Data for all cities saved to 'all_city_cars.csv'


In [55]:
all_city_cars_df.shape

(8369, 84)

Data Cleaning

In [57]:
all_city_cars_df=pd.read_csv(r"E:\GUVI Projects\Cardehko\all_city_cars.csv")
all_city_cars_df

C:\Users\vivsk\AppData\Local\Temp\ipykernel_21880\3446671135.py:1: DtypeWarning: Columns (74) have mixed types. Specify dtype option on import or set low_memory=False.
  all_city_cars_df=pd.read_csv(r"E:\GUVI Projects\Cardehko\all_city_cars.csv")


,Front Brake Type,Steering Type,Wheel Base,Wheel Size,BoreX Stroke,Color,trendingText.desc,Acceleration,Mileage,No Door Numbers,...,priceActual,car_links,Rear Brake Type,Drive Type,ownerNo,Height,centralVariantId,Kms Driven_value,Compression Ratio,Fuel Type_icon
0,Disc,Electric,2500,16,NaN,Black,High chances of sale in next 6 days,NaN,NaN,5.0,...,NaN,https://www.cardekho.com/used-car-details/used...,Drum,FWD,1,1642,8654,"20,000 Kms",NaN,https://images10.gaadi.com/listing/vdp/co/v1/f...
1,Ventilated Disc,Manual,2350mm,NaN,NaN,Grey,High chances of sale in next 6 days,15.7 Seconds,15.37 kmpl,5.0,...,NaN,https://www.cardekho.com/buy-used-car-details/...,Drum,RWD,1,1800mm,4025,"20,687 Kms",9.9:1,https://images10.gaadi.com/listing/vdp/co/v1/f...
2,Disc,Electronic,2500,16,72.2 x 81.3,Others,High chances of sale in next 6 days,11.7,20 kmpl,5.0,...,NaN,https://www.cardekho.com/used-car-details/used...,Drum,NaN,1,1572,8135,"30,000 Kms",NaN,https://images10.gaadi.com/listing/vdp/co/v1/f...
3,Ventilated Disc,Power,2380mm,NaN,NaN,Silver,High chances of sale in next 6 days,14.3 Seconds,19.81 kmpl,5.0,...,NaN,https://www.cardekho.com/buy-used-car-details/...,Drum,FWD,1,1550mm,1579,"59,247 Kms",NaN,https://images10.gaadi.com/listing/vdp/co/v1/f...
4,Disc,Power,2530mm,15,NaN,Others,High chances of sale in next 6 days,13.7 Seconds,18.7 kmpl,5.0,...,NaN,https://www.cardekho.com/used-car-details/used...,Drum,FWD,1,1544mm,1341,"50,000 Kms",NaN,https://images10.gaadi.com/listing/vdp/co/v1/f...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8364,Ventilated Disc,Electric,2435,NaN,NaN,Others,High chances of sale in next 6 days,NaN,25.24 kmpl,5.0,...,NaN,https://www.cardekho.com/used-car-details/used...,Drum,NaN,1,1555,8682,"10,000 Kms",NaN,https://images10.gaadi.com/listing/vdp/co/v1/f...
8365,Solid Disc,Power,2360mm,NaN,NaN,Others,High chances of sale in next 6 days,19 Seconds,22.74 kmpl,5.0,...,NaN,https://www.cardekho.com/used-car-details/used...,Drum,FWD,1,1475mm,3943,"1,20,000 Kms",NaN,https://images10.gaadi.com/listing/vdp/co/v1/f...
8366,Ventilated Disc,Power,2760mm,17,NaN,Others,High chances of sale in next 6 days,8.8 Seconds,11.74 kmpl,4.0,...,NaN,https://www.cardekho.com/used-car-details/used...,Solid Disc,RWD,3,1447mm,4672,"50,000 Kms",9.3:1,https://images10.gaadi.com/listing/vdp/co/v1/f...
8367,Ventilated Disc,Power,2360mm,14,NaN,Others,High chances of sale in next 6 days,15 Seconds,18.5 kmpl,5.0,...,NaN,https://www.cardekho.com/used-car-details/used...,Drum,FWD,1,1620mm,4144,"40,000 Kms",NaN,https://images10.gaadi.com/listing/vdp/co/v1/f...


In [58]:
all_city_cars_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8369 entries, 0 to 8368
Data columns (total 84 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Front Brake Type           8273 non-null   object 
 1   Steering Type              8114 non-null   object 
 2   Wheel Base                 8206 non-null   object 
 3   Wheel Size                 5386 non-null   object 
 4   BoreX Stroke               2405 non-null   object 
 5   Color                      8366 non-null   object 
 6   trendingText.desc          8369 non-null   object 
 7   Acceleration               4857 non-null   object 
 8   Mileage                    8082 non-null   object 
 9   No Door Numbers            8358 non-null   float64
 10  Engine Type                8074 non-null   object 
 11  it                         8369 non-null   int64  
 12  Seats_value                8363 non-null   object 
 13  Insurance Validity_icon    8365 non-null   objec

In [59]:
# Engine Displacement_value  and Engine_value are same column so drop any one column,
# Kms Driven_value   and  km   are same column so drop any one column
# owner,Ownership_value and ownerNo are same column so drop any TWO column,....Many Columns Like This and Droped the all icon columns
# Drop The Columns with more null values


# List of columns to drop
columns_to_drop = ['Ground Clearance Unladen', 'Value Configuration', 'trendingText.heading', 'Ownership_icon', 
                   'Top_Features', 'priceFixedText','Wheel Size','RTO_value','Cargo Volumn','Alloy Wheel Size', 'Kms Driven_value',
                   'Steering Type','it', 'Turning Radius', 'priceSaving', 'Kms Driven_icon', 'trendingText.desc', 'Engine Type', 
                   'car_links', 'Insurance Validity_icon', 'Seats_icon', 'Rear Tread', 'Super Charger','Wheel Base',
                   'Common_Icon', 'Common_Icon.1','Front Tread','BoreX Stroke', 'Drive Type', 'priceActual', 'Engine Displacement_icon', 
                   'Transmission_icon', 'Year of Manufacture_icon','Turbo Charger', 'Rear Brake Type', 'trendingText.imgUrl', 'Top Speed', 'RTO_icon', 
                   'Acceleration', 'Fuel Suppy System', 'Gross Weight','Height','Compression Ratio', 'Fuel Type_icon', 
                   'Registration Year_icon','Front Brake Type','Gear Box','owner','Max Power','Seating Capacity','Seats_value','Ownership_value',
                   'Kerb Weight','Fuel Type_value','Displacement','No Door Numbers','Values per Cylinder','Max Torque','centralVariantId',
                   'Transmission_value','Engine','Tyre Type','variantName','Year of Manufacture_value','Registration Year_value','No of Cylinder','Torque']

# Drop the columns from the DataFrame
all_city_cars_df = all_city_cars_df.drop(columns=columns_to_drop, errors='ignore')

In [60]:
all_city_cars_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8369 entries, 0 to 8368
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Color                      8366 non-null   object 
 1   Mileage                    8082 non-null   object 
 2   modelYear                  8369 non-null   int64  
 3   Length                     8290 non-null   object 
 4   price                      8369 non-null   object 
 5   Width                      8286 non-null   object 
 6   oem                        8369 non-null   object 
 7   transmission               8369 non-null   object 
 8   Seats                      8363 non-null   float64
 9   Engine Displacement_value  8365 non-null   object 
 10  bt                         8365 non-null   object 
 11  city                       8369 non-null   object 
 12  model                      8369 non-null   object 
 13  km                         8369 non-null   objec

In [61]:
all_city_cars_df

,Color,Mileage,modelYear,Length,price,Width,oem,transmission,Seats,Engine Displacement_value,bt,city,model,km,Insurance Validity_value,ft,ownerNo
0,Black,NaN,2022,3995mm,₹ 11.50 Lakh,1790,Kia,Automatic,5.0,998 cc,SUV,chennai,Kia Sonet,"20,000",Third Party insurance,Petrol,1
1,Grey,15.37 kmpl,2015,3675mm,₹ 4.15 Lakh,1475mm,Maruti,Manual,7.0,1196 cc,Minivans,chennai,Maruti Eeco,"20,687",Comprehensive,Petrol,1
2,Others,20 kmpl,2021,3994mm,₹ 7.50 Lakh,1758,Nissan,Manual,5.0,999 cc,SUV,chennai,Nissan Magnite,"30,000",Third Party insurance,Petrol,1
3,Silver,19.81 kmpl,2015,3585mm,₹ 3.98 Lakh,1595mm,Hyundai,Manual,5.0,1086 cc,Hatchback,chennai,Hyundai i10,"59,247",Comprehensive,Petrol,1
4,Others,18.7 kmpl,2015,3955mm,₹ 5.50 Lakh,1694mm,Honda,Manual,5.0,1199 cc,Hatchback,chennai,Honda Jazz,"50,000",Third Party insurance,Petrol,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8364,Others,25.24 kmpl,2022,3695mm,₹ 5.10 Lakh,1655,Maruti,Manual,5.0,998 cc,Hatchback,Kolkata,Maruti Celerio,"10,000",Third Party insurance,Petrol,1
8365,Others,22.74 kmpl,2014,3395mm,₹ 1.80 Lakh,1490mm,Maruti,Manual,5.0,796 cc,Hatchback,Kolkata,Maruti Alto 800,"1,20,000",Third Party insurance,Petrol,1
8366,Others,11.74 kmpl,2011,4591mm,₹ 5.50 Lakh,1770mm,Mercedes-Benz,Automatic,5.0,1796 cc,Sedan,Kolkata,Mercedes-Benz C-Class,"50,000",Third Party insurance,Petrol,3
8367,Others,18.5 kmpl,2012,3775mm,₹ 1.40 Lakh,1680mm,Maruti,Manual,5.0,1197 cc,Hatchback,Kolkata,Maruti Ritz,"40,000",Third Party insurance,Petrol,1


Missing values

In [62]:
missing_values = all_city_cars_df.isnull().sum()
print(missing_values[missing_values > 0])

Color                          3
Mileage                      287
Length                        79
Width                         83
Seats                          6
Engine Displacement_value      4
bt                             4
Insurance Validity_value       4
dtype: int64


In [63]:
# Fill missing 'bt' (body type) values by propagating values within the same 'model'
all_city_cars_df['bt'] = all_city_cars_df.groupby('model')['bt'].transform(lambda group: group.ffill().bfill())


# If any missing values remain, fill them with the most common 'bt' value
all_city_cars_df['bt'].fillna(all_city_cars_df['bt'].mode()[0], inplace=True)

C:\Users\vivsk\AppData\Local\Temp\ipykernel_21880\596607894.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  all_city_cars_df['bt'] = all_city_cars_df.groupby('model')['bt'].transform(lambda group: group.ffill().bfill())
C:\Users\vivsk\AppData\Local\Temp\ipykernel_21880\596607894.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on t

In [64]:
# Fill missing 'Seats_value values by propagating values within the same 'model'
all_city_cars_df['Seats'] = all_city_cars_df.groupby('model')['Seats'].transform(lambda group: group.ffill().bfill())
# Handling remaining columns as needed
all_city_cars_df['Engine Displacement_value']=all_city_cars_df.groupby('model')['Engine Displacement_value'].transform(lambda group: group.ffill().bfill())
all_city_cars_df['Insurance Validity_value'].fillna(all_city_cars_df['Insurance Validity_value'].mode()[0], inplace=True)

C:\Users\vivsk\AppData\Local\Temp\ipykernel_21880\3254742270.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  all_city_cars_df['Insurance Validity_value'].fillna(all_city_cars_df['Insurance Validity_value'].mode()[0], inplace=True)


In [65]:
# Fill missing 'Mileage_value'  values by propagating values within the same 'model'
all_city_cars_df['Mileage_value(Kmpl)'] = all_city_cars_df.groupby('model')['Mileage'].transform(lambda group: group.ffill().bfill())
all_city_cars_df.drop(columns='Mileage',inplace=True)

all_city_cars_df['Length'] = all_city_cars_df.groupby('model')['Length'].transform(lambda group: group.ffill().bfill())
all_city_cars_df['Width'] = all_city_cars_df.groupby('model')['Width'].transform(lambda group: group.ffill().bfill())

C:\Users\vivsk\AppData\Local\Temp\ipykernel_21880\1083307966.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  all_city_cars_df['Mileage_value(Kmpl)'] = all_city_cars_df.groupby('model')['Mileage'].transform(lambda group: group.ffill().bfill())
C:\Users\vivsk\AppData\Local\Temp\ipykernel_21880\1083307966.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  all_city_cars_df['Length'] = all_city_cars_df.groupby('model')['Length'].transform(lambda group: group.ffill().bfill())
C:\Users\vivsk\AppData\Local\Temp\ipykernel_21880\1083307966.py:6: FutureWarning

In [66]:
missing_values = all_city_cars_df.isnull().sum()
print(missing_values[missing_values > 0]) 

Color                   3
Length                 10
Width                  10
Seats                   3
Mileage_value(Kmpl)    76
dtype: int64


In [67]:
# Remove rows with null values in specified columns
all_city_cars_df.dropna(subset='Mileage_value(Kmpl)',inplace=True)

Std data formats

In [68]:
# Ensure 'km' is treated as a string, remove commas, and convert to integer safely
all_city_cars_df['km'] = all_city_cars_df['km'].astype(str).str.replace(',', '')

# Use pd.to_numeric to convert, coercing errors to NaN
all_city_cars_df['km'] = pd.to_numeric(all_city_cars_df['km'], errors='coerce')
all_city_cars_df.head()

,Color,modelYear,Length,price,Width,oem,transmission,Seats,Engine Displacement_value,bt,city,model,km,Insurance Validity_value,ft,ownerNo,Mileage_value(Kmpl)
0,Black,2022,3995mm,₹ 11.50 Lakh,1790,Kia,Automatic,5.0,998 cc,SUV,chennai,Kia Sonet,20000,Third Party insurance,Petrol,1,18.2 kmpl
1,Grey,2015,3675mm,₹ 4.15 Lakh,1475mm,Maruti,Manual,7.0,1196 cc,Minivans,chennai,Maruti Eeco,20687,Comprehensive,Petrol,1,15.37 kmpl
2,Others,2021,3994mm,₹ 7.50 Lakh,1758,Nissan,Manual,5.0,999 cc,SUV,chennai,Nissan Magnite,30000,Third Party insurance,Petrol,1,20 kmpl
3,Silver,2015,3585mm,₹ 3.98 Lakh,1595mm,Hyundai,Manual,5.0,1086 cc,Hatchback,chennai,Hyundai i10,59247,Comprehensive,Petrol,1,19.81 kmpl
4,Others,2015,3955mm,₹ 5.50 Lakh,1694mm,Honda,Manual,5.0,1199 cc,Hatchback,chennai,Honda Jazz,50000,Third Party insurance,Petrol,1,18.7 kmpl


In [69]:
def convert_numeric(value):
    new = value.replace("₹", "").replace(',', '').strip()
    if "Lakh" in new:
        return float(new.replace('Lakh', '').strip()) * 1e5
    elif "Crore" in new:
        return float(new.replace("Crore", "").strip()) * 1e7
    else:
        return float(new)

# Apply the function to the 'price' column correctly
all_city_cars_df["price"] = all_city_cars_df["price"].apply(convert_numeric)

In [70]:
all_city_cars_df.head()

,Color,modelYear,Length,price,Width,oem,transmission,Seats,Engine Displacement_value,bt,city,model,km,Insurance Validity_value,ft,ownerNo,Mileage_value(Kmpl)
0,Black,2022,3995mm,1150000.0,1790,Kia,Automatic,5.0,998 cc,SUV,chennai,Kia Sonet,20000,Third Party insurance,Petrol,1,18.2 kmpl
1,Grey,2015,3675mm,415000.0,1475mm,Maruti,Manual,7.0,1196 cc,Minivans,chennai,Maruti Eeco,20687,Comprehensive,Petrol,1,15.37 kmpl
2,Others,2021,3994mm,750000.0,1758,Nissan,Manual,5.0,999 cc,SUV,chennai,Nissan Magnite,30000,Third Party insurance,Petrol,1,20 kmpl
3,Silver,2015,3585mm,398000.0,1595mm,Hyundai,Manual,5.0,1086 cc,Hatchback,chennai,Hyundai i10,59247,Comprehensive,Petrol,1,19.81 kmpl
4,Others,2015,3955mm,550000.0,1694mm,Honda,Manual,5.0,1199 cc,Hatchback,chennai,Honda Jazz,50000,Third Party insurance,Petrol,1,18.7 kmpl


In [71]:
def extract_cc(value):
    if isinstance(value, str):  # Check if the value is a string
        try:
            return int(value.split()[0])  # Extract the first part and convert to int
        except (ValueError, IndexError):
            return None
    return None  # Return None for non-string (e.g., float) values

# Apply the function to extract seat values
all_city_cars_df['Engine Displacement_value(cc)'] = all_city_cars_df['Engine Displacement_value'].apply(extract_cc)

# Convert the 'Engine Displacement_value in cc' column to integers
all_city_cars_df['Engine Displacement_value(cc)' ] = all_city_cars_df['Engine Displacement_value(cc)'].astype(int)

# Drop the old 'Engine Displacement_value' column 
all_city_cars_df.drop('Engine Displacement_value', axis=1, inplace=True)

In [72]:

all_city_cars_df.head()

,Color,modelYear,Length,price,Width,oem,transmission,Seats,bt,city,model,km,Insurance Validity_value,ft,ownerNo,Mileage_value(Kmpl),Engine Displacement_value(cc)
0,Black,2022,3995mm,1150000.0,1790,Kia,Automatic,5.0,SUV,chennai,Kia Sonet,20000,Third Party insurance,Petrol,1,18.2 kmpl,998
1,Grey,2015,3675mm,415000.0,1475mm,Maruti,Manual,7.0,Minivans,chennai,Maruti Eeco,20687,Comprehensive,Petrol,1,15.37 kmpl,1196
2,Others,2021,3994mm,750000.0,1758,Nissan,Manual,5.0,SUV,chennai,Nissan Magnite,30000,Third Party insurance,Petrol,1,20 kmpl,999
3,Silver,2015,3585mm,398000.0,1595mm,Hyundai,Manual,5.0,Hatchback,chennai,Hyundai i10,59247,Comprehensive,Petrol,1,19.81 kmpl,1086
4,Others,2015,3955mm,550000.0,1694mm,Honda,Manual,5.0,Hatchback,chennai,Honda Jazz,50000,Third Party insurance,Petrol,1,18.7 kmpl,1199


In [73]:
# Convert the 'Year of Manufacture_value ' column to integers
all_city_cars_df['modelYear'] = all_city_cars_df['modelYear'].astype(int)
all_city_cars_df.head()

,Color,modelYear,Length,price,Width,oem,transmission,Seats,bt,city,model,km,Insurance Validity_value,ft,ownerNo,Mileage_value(Kmpl),Engine Displacement_value(cc)
0,Black,2022,3995mm,1150000.0,1790,Kia,Automatic,5.0,SUV,chennai,Kia Sonet,20000,Third Party insurance,Petrol,1,18.2 kmpl,998
1,Grey,2015,3675mm,415000.0,1475mm,Maruti,Manual,7.0,Minivans,chennai,Maruti Eeco,20687,Comprehensive,Petrol,1,15.37 kmpl,1196
2,Others,2021,3994mm,750000.0,1758,Nissan,Manual,5.0,SUV,chennai,Nissan Magnite,30000,Third Party insurance,Petrol,1,20 kmpl,999
3,Silver,2015,3585mm,398000.0,1595mm,Hyundai,Manual,5.0,Hatchback,chennai,Hyundai i10,59247,Comprehensive,Petrol,1,19.81 kmpl,1086
4,Others,2015,3955mm,550000.0,1694mm,Honda,Manual,5.0,Hatchback,chennai,Honda Jazz,50000,Third Party insurance,Petrol,1,18.7 kmpl,1199


In [74]:
# Function to clean and normalize mileage values
def clean_mileage(value):
    # Check for blank or empty values
    if pd.isna(value) or str(value).strip() == '':
        return None  # Return None or 0 as preferred

    # Match numerical value with optional unit (kmpl)
    match = re.search(r"(\d*\.?\d+)\s*(kmpl)?", str(value), re.IGNORECASE)
    
    if match:
        return round(float(match.group(1)), 2)  # Return the cleaned and rounded value
    return None

# Apply the cleaning function to the 'Mileage' column
all_city_cars_df['Mileage_value(Kmpl)'] = all_city_cars_df['Mileage_value(Kmpl)'].apply(clean_mileage)

# Drop the old column
all_city_cars_df.drop('Mileage_value(Kmpl)', axis=1, inplace=True)

In [75]:
all_city_cars_df.head()

,Color,modelYear,Length,price,Width,oem,transmission,Seats,bt,city,model,km,Insurance Validity_value,ft,ownerNo,Engine Displacement_value(cc)
0,Black,2022,3995mm,1150000.0,1790,Kia,Automatic,5.0,SUV,chennai,Kia Sonet,20000,Third Party insurance,Petrol,1,998
1,Grey,2015,3675mm,415000.0,1475mm,Maruti,Manual,7.0,Minivans,chennai,Maruti Eeco,20687,Comprehensive,Petrol,1,1196
2,Others,2021,3994mm,750000.0,1758,Nissan,Manual,5.0,SUV,chennai,Nissan Magnite,30000,Third Party insurance,Petrol,1,999
3,Silver,2015,3585mm,398000.0,1595mm,Hyundai,Manual,5.0,Hatchback,chennai,Hyundai i10,59247,Comprehensive,Petrol,1,1086
4,Others,2015,3955mm,550000.0,1694mm,Honda,Manual,5.0,Hatchback,chennai,Honda Jazz,50000,Third Party insurance,Petrol,1,1199


In [76]:
# Example: Rename specific columns using a dictionary
all_city_cars_df = all_city_cars_df.rename(columns={
    'ft':'Fuel type',
    'bt':'Body type',
    'km':'kilometer driven',
    'transmission':'Transmission type',
    'ownerNo':'Number of previous owners',
    'oem':'Brand Name',
    'model':'Car model',
    'modelYear':'Year of car manufacture'
})

print(all_city_cars_df.head())

    Color  Year of car manufacture  Length      price   Width Brand Name  \
0   Black                     2022  3995mm  1150000.0    1790        Kia   
1    Grey                     2015  3675mm   415000.0  1475mm     Maruti   
2  Others                     2021  3994mm   750000.0    1758     Nissan   
3  Silver                     2015  3585mm   398000.0  1595mm    Hyundai   
4  Others                     2015  3955mm   550000.0  1694mm      Honda   

  Transmission type  Seats  Body type     city       Car model  \
0         Automatic    5.0        SUV  chennai       Kia Sonet   
1            Manual    7.0   Minivans  chennai     Maruti Eeco   
2            Manual    5.0        SUV  chennai  Nissan Magnite   
3            Manual    5.0  Hatchback  chennai     Hyundai i10   
4            Manual    5.0  Hatchback  chennai      Honda Jazz   

   kilometer driven Insurance Validity_value Fuel type  \
0             20000    Third Party insurance    Petrol   
1             20687           

In [77]:
all_city_cars_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8293 entries, 0 to 8368
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Color                          8290 non-null   object 
 1   Year of car manufacture        8293 non-null   int64  
 2   Length                         8286 non-null   object 
 3   price                          8293 non-null   float64
 4   Width                          8286 non-null   object 
 5   Brand Name                     8293 non-null   object 
 6   Transmission type              8293 non-null   object 
 7   Seats                          8290 non-null   float64
 8   Body type                      8293 non-null   object 
 9   city                           8293 non-null   object 
 10  Car model                      8293 non-null   object 
 11  kilometer driven               8293 non-null   int64  
 12  Insurance Validity_value       8293 non-null   object

In [78]:
# Drop columns where all values are NaN
all_city_cars_df = all_city_cars_df.dropna(axis=1, how='all')

print(all_city_cars_df)

       Color  Year of car manufacture  Length      price   Width  \
0      Black                     2022  3995mm  1150000.0    1790   
1       Grey                     2015  3675mm   415000.0  1475mm   
2     Others                     2021  3994mm   750000.0    1758   
3     Silver                     2015  3585mm   398000.0  1595mm   
4     Others                     2015  3955mm   550000.0  1694mm   
...      ...                      ...     ...        ...     ...   
8364  Others                     2022  3695mm   510000.0    1655   
8365  Others                     2014  3395mm   180000.0  1490mm   
8366  Others                     2011  4591mm   550000.0  1770mm   
8367  Others                     2012  3775mm   140000.0  1680mm   
8368  Others                     2017  4315mm   500000.0  1822mm   

         Brand Name Transmission type  Seats  Body type     city  \
0               Kia         Automatic    5.0        SUV  chennai   
1            Maruti            Manual    7.0   

In [79]:
# Drop rows where any value is NaN
all_city_cars_df = all_city_cars_df.dropna()

print(all_city_cars_df.info())  # To check how many rows remain after dropping

<class 'pandas.core.frame.DataFrame'>
Index: 8283 entries, 0 to 8368
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Color                          8283 non-null   object 
 1   Year of car manufacture        8283 non-null   int64  
 2   Length                         8283 non-null   object 
 3   price                          8283 non-null   float64
 4   Width                          8283 non-null   object 
 5   Brand Name                     8283 non-null   object 
 6   Transmission type              8283 non-null   object 
 7   Seats                          8283 non-null   float64
 8   Body type                      8283 non-null   object 
 9   city                           8283 non-null   object 
 10  Car model                      8283 non-null   object 
 11  kilometer driven               8283 non-null   int64  
 12  Insurance Validity_value       8283 non-null   object

In [80]:
all_city_cars_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8283 entries, 0 to 8368
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Color                          8283 non-null   object 
 1   Year of car manufacture        8283 non-null   int64  
 2   Length                         8283 non-null   object 
 3   price                          8283 non-null   float64
 4   Width                          8283 non-null   object 
 5   Brand Name                     8283 non-null   object 
 6   Transmission type              8283 non-null   object 
 7   Seats                          8283 non-null   float64
 8   Body type                      8283 non-null   object 
 9   city                           8283 non-null   object 
 10  Car model                      8283 non-null   object 
 11  kilometer driven               8283 non-null   int64  
 12  Insurance Validity_value       8283 non-null   object

In [81]:
all_city_cars_df.to_csv('all_car_details.csv')